In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [27]:
filepath = "/content/drive/MyDrive/data/enrollment/data/Enrollment.xlsx"

training_set = pd.read_excel(filepath,sheet_name='data')
holdout = pd.read_excel(filepath,sheet_name='holdhout')

In [28]:
training_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5382 entries, 0 to 5381
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   FirstGen                550 non-null    object        
 1   sat_S11                 853 non-null    float64       
 2   SAT_S12                 852 non-null    float64       
 3   Gender                  5382 non-null   object        
 4   Citizen                 5372 non-null   object        
 5   StudentType             5382 non-null   int64         
 6   MAJOR1                  5382 non-null   object        
 7   MAJOR2                  760 non-null    object        
 8   highschool_desc         5372 non-null   object        
 9   street                  5368 non-null   object        
 10  city                    5368 non-null   object        
 11  state                   4972 non-null   object        
 12  zip                     5308 non-null   object  

In [54]:
df = training_set.copy()

# FirstGen
df['FirstGen'] = df['FirstGen'].fillna(0).apply(lambda x: 0 if x == 0 else 1)

# Gender
df['Female'] = df['Gender'].apply(lambda x: 1 if x == 'F' else 0)
df['No Gender'] = df['Gender'].apply(lambda x: 1 if x == 'N' else 0)
df['Male'] = df['Gender'].apply(lambda x: 1 if x == 'M' else 0)
df = df.drop('Gender',axis=1)

# Citizen
df['Citizen'] = df['Citizen'].dropna().apply(lambda x: 1 if x == 'Y' else 0)

# StudentType
df['New Student'] = df['StudentType'].apply(lambda x: 1 if x == 5 else 0)
df = df.drop('StudentType',axis=1)

# Major
df = df.drop('MAJOR1',axis=1)
df['Double Major'] = df['MAJOR2'].fillna(0).apply(lambda x: 0 if x == 0 else 1)
df = df.drop('MAJOR2',axis=1)

# highschool_desc
df = pd.get_dummies(df, columns=['highschool_desc'], prefix='highschool')

# address
df = df.drop(['street','city'],axis=1)

# state
df['CA Resident'] = df['state'].dropna().apply(lambda x: 1 if x == 'CA' else 0)
df = df.drop(['state','zip'],axis=1)

# RELIGION
no_affiliation = ['No affliation/none listed']
sda = ['Seventh-day Adventist']
christian = ['Christian', 'Baptist', 'Lutheran', 'Latter-day Saints',
                  'Christian-Disc of Christ', 'Methodist', 'Presbyterian',
                  'First Christian', 'Protestant', 'Church of God',
                  'Evangelical', 'Pentecostal', 'Apostolic', 'Church of Christ',
                  'Nazarene', 'Assembly of God', 'African-Methodist-Episcopal',
                  'United Church of Christ', 'Anglican', 'Congregational',
                  'Southern Baptist']
catholic = ['Roman Catholic', 'Eastern Orthodox', 'Coptic Orthodox']
other = [
    'Agnostic', 'Jainism', 'Muslim', 'Sikh', 'Hindu', 'Buddhist',
    'Jehovahs Witnesses', 'Nichiren Shosshu of America', 'Episcopalian',
    'Unification', 'Friends/Quaker']

df['RELIGION_No_Affiliation'] = df['RELIGION'].apply(lambda x: 1 if x in no_affiliation else 0)
df['RELIGION_SDA'] = df['RELIGION'].apply(lambda x: 1 if x in sda else 0)
df['RELIGION_Christian'] = df['RELIGION'].apply(lambda x: 1 if x in christian else 0)
df['RELIGION_Catholic'] = df['RELIGION'].apply(lambda x: 1 if x in catholic else 0)
df['RELIGION_Other'] = df['RELIGION'].apply(lambda x: 1 if x in other else 0)
df = df.drop('RELIGION',axis=1)

# NETHN
df = pd.get_dummies(df, columns=['NETHN'], prefix='NETHN')

# RACE
unique_races = {'B', 'N', 'W', 'A', 'AI'} # Black, Native Hawaiian, White, Asian, American Indian
for race in unique_races:
    df[f'RACE_{race}'] = df['RACE'].dropna().apply(lambda x: 1 if race in x else 0)
df = df.drop('RACE', axis=1)

# GPA
df = df[df['HighSchool_GPA'] != 328]

# PERCENTILE
df = df.drop('HighSchool_PERCENTILE',axis=1) # drop as recommended by Dr. Karina

# CLASS_RANK
# --- nothing to change ---

# NumDays_app_adm
df = df[df['NumDays_app_adm'] >= 0]

# admit_deg_code
df['Pre Professional'] = df['admit_deg_code'].apply(lambda x: 1 if x == 3 else 0)
df = pd.get_dummies(df, columns=['admit_deg_code'], prefix = 'admit_deg_code')
df = df.drop(['admit_deg_code_49.0','admit_deg_code_50.0'],axis=1)

# drop not useful columns
df = df.drop(['Application Ineligible', 'Application Entered', 'Admitted', 'Enrolled', 'enrolled_fall_2024', 'enrolled_fall_2023', 'enrolled_fall_2022', 'enrolled_fall_2021'], axis=1)


In [55]:
correlation = df.corr()['Enrolled2'].sort_values(ascending=False)

print(correlation)


Enrolled2                                    1.000000
highschool_SDA HS-PU Conf Constituent        0.311556
RELIGION_SDA                                 0.196379
CA Resident                                  0.176600
RACE_A                                       0.112869
Male                                         0.068852
SAT_S12                                      0.063498
highschool_SDA HS-PU Conf Non-Constituent    0.057178
NETHN_2.0                                    0.052966
sat_S11                                      0.050858
RACE_N                                       0.047081
highschool_HS Equiv/Home Sch State Req US    0.045196
NETHN_1.0                                    0.042951
admit_deg_code_50.0                          0.041627
Citizen                                      0.039331
Pre Professional                             0.038531
admit_deg_code_3.0                           0.038531
admit_deg_code_49.0                          0.032930
admit_deg_code_0.0          

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5374 entries, 0 to 5381
Data columns (total 63 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   FirstGen                                   5374 non-null   int64         
 1   sat_S11                                    851 non-null    float64       
 2   SAT_S12                                    850 non-null    float64       
 3   Citizen                                    5364 non-null   float64       
 4   HighSchool_GPA                             4906 non-null   float64       
 5   HighSchool_CLASS_RANK                      3319 non-null   float64       
 6   Application Ineligible                     16 non-null     datetime64[ns]
 7   Application Entered                        5374 non-null   datetime64[ns]
 8   Admitted                                   5374 non-null   datetime64[ns]
 9   NumDays_app_adm         

In [38]:
correlations = df[['FirstGen','Female','No Gender', 'Male', 'CA Resident', 'Citizen', 'New Student', 'Pre Professional', 'HighSchool_GPA', 'HighSchool_PERCENTILE', 'Enrolled2']].corr()['Enrolled2'].sort_values(ascending=False)
print(correlations)

Enrolled2                1.000000
CA Resident              0.176491
Male                     0.068572
Citizen                  0.039465
Pre Professional         0.038379
FirstGen                -0.001428
No Gender               -0.013405
HighSchool_GPA          -0.062207
Female                  -0.067893
HighSchool_PERCENTILE   -0.149406
New Student             -0.250420
Name: Enrolled2, dtype: float64


array([ 1.,  0., nan])